In [1]:
!pip install ipywidgets

1.IMPORTS

In [2]:
import cv2
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.applications.efficientnet import preprocess_input
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from io import BytesIO

print("✅ All libraries imported successfully!")


d:\anaconda\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


✅ All libraries imported successfully!


2. LOAD MODEL

In [3]:
print("Loading model...")
model = tf.keras.models.load_model('Models/EfficientNetB3/efficientnetb3_model.keras')
model.build((None, 224, 224, 3))
print("✅ Model loaded successfully!")

# Define class names
class_names = [
    'Pepper Bell - Bacterial Spot',
    'Pepper Bell - Healthy',
    'Potato - Early Blight',
    'Potato - Late Blight',
    'Potato - Healthy',
    'Tomato - Target Spot',
    'Tomato - Yellow Leaf Curl Virus',
    'Tomato - Mosaic Virus',
    'Tomato - Healthy'
]

# Disease information
class_info = {
    'Pepper Bell - Bacterial Spot': '🌶️ Bacterial disease causing dark spots on leaves and fruits.',
    'Pepper Bell - Healthy': '✅ Your pepper plant looks healthy!',
    'Potato - Early Blight': '🥔 Fungal disease causing dark spots with concentric rings.',
    'Potato - Late Blight': '🥔 Serious fungal disease that can destroy entire crops.',
    'Potato - Healthy': '✅ Your potato plant looks healthy!',
    'Tomato - Target Spot': '🍅 Fungal disease causing circular spots with dark centers.',
    'Tomato - Yellow Leaf Curl Virus': '🍅 Viral disease causing yellowing and curling of leaves.',
    'Tomato - Mosaic Virus': '🍅 Viral disease causing mottled pattern on leaves.',
    'Tomato - Healthy': '✅ Your tomato plant looks healthy!'
}

# Get last conv layer
base_model = model.get_layer('efficientnetb3')
last_conv_layer_name = None
for layer in reversed(base_model.layers):
    if 'conv' in layer.name.lower():
        last_conv_layer_name = layer.name
        break

print(f"🎯 Using layer: {last_conv_layer_name}")

Loading model...
✅ Model loaded successfully!
🎯 Using layer: top_conv


3. DEFINE GRAD-CAM FUNCTIONS

In [4]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    """Generate Grad-CAM heatmap"""
    base_model = model.get_layer('efficientnetb3')
    last_conv_layer = base_model.get_layer(last_conv_layer_name)
    
    grad_model = tf.keras.Model(
        inputs=[base_model.input],
        outputs=[last_conv_layer.output, base_model.output]
    )
    
    with tf.GradientTape() as tape:
        last_conv_output, base_predictions = grad_model(img_array)
        
        x = last_conv_output
        for layer in model.layers[1:]:
            x = layer(x)
        predictions = x
        
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    
    grads = tape.gradient(class_channel, last_conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_output = last_conv_output[0]
    heatmap = last_conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    
    return heatmap.numpy()

def overlay_gradcam(img, heatmap, alpha=0.4):
    """Overlay heatmap on image"""
    img = cv2.resize(np.array(img), (224, 224))
    heatmap = cv2.resize(heatmap, (224, 224))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    superimposed = cv2.addWeighted(img, 1-alpha, heatmap, alpha, 0)
    return superimposed

print("✅ Grad-CAM functions defined!")


✅ Grad-CAM functions defined!


4. DEFINE PREDICTION FUNCTION

In [5]:
def predict_and_visualize(image):
    """Predict disease and create visualization"""
    # Preprocess
    img = image.resize((224, 224))
    img_array = img_to_array(img)
    img_array_preprocessed = preprocess_input(np.expand_dims(img_array, axis=0))
    
    # Predict
    predictions = model.predict(img_array_preprocessed, verbose=0)
    predicted_idx = np.argmax(predictions[0])
    confidence = np.max(predictions[0])
    predicted_class = class_names[predicted_idx]
    
    # Generate Grad-CAM
    heatmap = make_gradcam_heatmap(img_array_preprocessed, model, last_conv_layer_name)
    gradcam_img = overlay_gradcam(img, heatmap)
    
    return predicted_class, confidence, predictions[0], heatmap, img_array, gradcam_img

print("✅ Prediction function defined!")


✅ Prediction function defined!


5. CREATE INTERACTIVE GUI

In [6]:
# Create widgets
upload_widget = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload Image',
    button_style='success',
    style={'description_width': 'initial'}
)

analyze_button = widgets.Button(
    description='🔍 Analyze Image',
    button_style='info',
    layout=widgets.Layout(width='200px', height='40px')
)

output_area = widgets.Output()

# Display header
display(HTML("""
<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
            padding: 30px; 
            border-radius: 15px; 
            text-align: center;
            margin-bottom: 20px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);'>
    <h1 style='color: white; margin: 0; font-size: 36px;'>🌱 Plant Disease Detection System</h1>
    <p style='color: #e0e0e0; margin: 10px 0 0 0; font-size: 16px;'>
        Powered by EfficientNetB3 & Grad-CAM Visualization
    </p>
</div>
"""))

display(HTML("""
<div style='background-color: #f8f9fa; 
            padding: 20px; 
            border-radius: 10px; 
            margin-bottom: 20px;
            border-left: 5px solid #667eea;'>
    <h3 style='margin-top: 0;'>📋 Instructions:</h3>
    <ol style='line-height: 1.8;'>
        <li>Click the <strong>"Upload Image"</strong> button below</li>
        <li>Select a plant leaf image (JPG, PNG, JPEG)</li>
        <li>Click <strong>"🔍 Analyze Image"</strong> to get results</li>
        <li>View the prediction, confidence, and Grad-CAM visualization</li>
    </ol>
    <p><strong>Supported Plants:</strong> 🌶️ Pepper Bell | 🥔 Potato | 🍅 Tomato</p>
</div>
"""))

# Display upload widget and button
display(upload_widget)
display(analyze_button)
display(output_area)

# Define button click handler
def on_analyze_click(b):
    with output_area:
        clear_output(wait=True)
        
        if not upload_widget.value:
            display(HTML("""
            <div style='background-color: #fff3cd; 
                        padding: 15px; 
                        border-radius: 8px; 
                        border-left: 4px solid #ffc107;'>
                ⚠️ <strong>Please upload an image first!</strong>
            </div>
            """))
            return
        
        try:
            # Get uploaded image
            uploaded_file = list(upload_widget.value.values())[0]
            image = Image.open(BytesIO(uploaded_file['content']))
            
            display(HTML("<h2>🔄 Analyzing image...</h2>"))
            
            # Make prediction
            pred_class, confidence, all_preds, heatmap, img_array, gradcam_img = predict_and_visualize(image)
            
            clear_output(wait=True)
            
            # Display results header
            display(HTML("<h2 style='color: #667eea;'>📊 Analysis Results</h2>"))
            
            # Prediction box
            if "Healthy" in pred_class:
                box_color = "#d4edda"
                border_color = "#28a745"
                icon = "✅"
            else:
                box_color = "#fff3cd"
                border_color = "#ffc107"
                icon = "⚠️"
            
            display(HTML(f"""
            <div style='background-color: {box_color}; 
                        padding: 20px; 
                        border-radius: 10px; 
                        margin: 20px 0;
                        border-left: 5px solid {border_color};'>
                <h2 style='margin: 0 0 10px 0;'>{icon} Prediction: {pred_class}</h2>
                <h3 style='margin: 0;'>📈 Confidence: {confidence*100:.2f}%</h3>
                <p style='margin: 10px 0 0 0; font-size: 16px;'>{class_info[pred_class]}</p>
            </div>
            """))
            
            # Visualizations
            display(HTML("<h3 style='color: #667eea;'>🔥 Grad-CAM Visualization</h3>"))
            display(HTML("<p style='color: #666;'><em>The heatmap shows which parts of the image the model focused on for its prediction.</em></p>"))
            
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            
            # Original
            axes[0].imshow(img_array.astype('uint8'))
            axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
            axes[0].axis('off')
            
            # Heatmap
            axes[1].imshow(heatmap, cmap='jet')
            axes[1].set_title('Grad-CAM Heatmap', fontsize=14, fontweight='bold')
            axes[1].axis('off')
            
            # Overlay
            axes[2].imshow(cv2.cvtColor(gradcam_img, cv2.COLOR_BGR2RGB))
            axes[2].set_title('Overlay', fontsize=14, fontweight='bold')
            axes[2].axis('off')
            
            plt.tight_layout()
            plt.show()
            
            # All predictions bar chart
            display(HTML("<h3 style='color: #667eea; margin-top: 30px;'>📈 Confidence Scores for All Classes</h3>"))
            
            fig, ax = plt.subplots(figsize=(10, 6))
            colors = ['#28a745' if i == np.argmax(all_preds) else '#667eea' for i in range(len(all_preds))]
            bars = ax.barh(class_names, all_preds * 100, color=colors, alpha=0.8)
            
            ax.set_xlabel('Confidence (%)', fontsize=12, fontweight='bold')
            ax.set_title('Prediction Confidence for All Classes', fontsize=14, fontweight='bold')
            ax.grid(axis='x', alpha=0.3)
            
            # Add value labels
            for i, (bar, val) in enumerate(zip(bars, all_preds * 100)):
                ax.text(val + 1, i, f'{val:.1f}%', va='center', fontsize=10)
            
            plt.tight_layout()
            plt.show()
            
            # Success message
            display(HTML("""
            <div style='background-color: #d4edda; 
                        padding: 15px; 
                        border-radius: 8px; 
                        margin-top: 20px;
                        border-left: 4px solid #28a745;'>
                ✅ <strong>Analysis complete!</strong> Upload another image to analyze more plants.
            </div>
            """))
            
        except Exception as e:
            display(HTML(f"""
            <div style='background-color: #f8d7da; 
                        padding: 15px; 
                        border-radius: 8px; 
                        border-left: 4px solid #dc3545;'>
                ❌ <strong>Error:</strong> {str(e)}
            </div>
            """))

# Connect button to handler
analyze_button.on_click(on_analyze_click)

print("✅ Upload an image and click Analyze!")


FileUpload(value=(), accept='image/*', button_style='success', description='Upload Image')

Button(button_style='info', description='🔍 Analyze Image', layout=Layout(height='40px', width='200px'), style=…

Output()

✅ Upload an image and click Analyze!
